**[🏠 Course Home](../README.md) | 🐍 Python Companion: [Appendix F: Flake Culprit Search](../python/appendix_information_theory_and_culprit_search.ipynb) | 🔵 R Companion: [Appendix F: Flake Culprit Search](../r/appendix_information_theory_and_culprit_search.ipynb)**

---

# Appendix F: Active Learning & Flake Culprit Search
### *From Shannon Entropy and Mutual Information to Bayesian Changepoint Localization in Flaky CI Pipelines*

> **Executive Overview**: In continuous integration (CI) environments,
> software regressions do not always manifest as deterministic failures
> ($0\% \to 100\%$). Often, a subtle race condition, memory leak, or timing
> dependency causes a **flaky test regression**, where the test failure rate
> jumps from $1\%$ to $25\%$. When this happens, classical debugging algorithms
> like `git bisect` fail catastrophically: a single accidental pass at a broken
> revision causes binary search to permanently eliminate the true culprit from
> consideration.
>
> To solve this without brute-force testing thousands of revisions, production
> infrastructure systems combine two foundational mathematical pillars:
> 1. **Bayesian Changepoint Estimation**: An exact $\mathcal{O}(N)$
>    Beta-Bernoulli conjugate marginal likelihood model that calculates the
>    full posterior probability distribution over all suspect commits.
> 2. **Information Theory & Active Learning**: Using **Shannon Entropy**,
>    **Conditional Entropy**, and **Mutual Information** to calculate the
>    *expected information gain* of testing any revision, ensuring that every
>    expensive CI build destroys the maximum possible volume of uncertainty.

---

## 1. The Flaky Bisect Trap (Why Binary Search Fails)

In traditional software development, finding a regression is treated as a
deterministic binary search:
- You know revision $0$ was good ($0\%$ failure).
- You know revision $100$ is bad ($100\%$ failure).
- You test revision $50$:
  - If it fails, the regression happened in $[0, 50]$.
  - If it passes, the regression happened in $[51, 100]$.
- In $\log_2(100) \approx 7$ builds, you pinpoint the exact commit.

### The Foggy Swamp of Flakiness
Now consider reality: commit 42 introduced a race condition that causes a test
to fail $25\%$ of the time ($p_{\text{after}} = 0.25$). Prior to commit 42,
background infra flakes caused the test to fail $1\%$ of the time
($p_{\text{before}} = 0.01$).

If you run standard `git bisect` and test revision $50$ **once**:
* Revision 50 is *after* the culprit (failure rate $25\%$).
* But that means it has a **$75\%$ chance of passing** on any single run!
* The moment revision 50 passes, `git bisect` proudly concludes:
  *"The regression must be in commits 51 to 100!"*
* It permanently discards commits $0$ through $50$. The true culprit (commit 42)
  is erased from consideration. The bisect has failed with $75\%$ probability on
  step 1 alone.

### The Brute-Force Fallacy
To fix this, an engineer might suggest:
*"Just run 100 iterations at each step!"*
If each build takes 15 minutes and costs compute quota, running 100 builds per
bisect step across 7 steps requires **700 builds**. Worse, running 100 builds
across all 100 candidate commits requires **10,000 builds**.

Can we locate the culprit with **high statistical confidence ($\ge 99\%$)**
using only a tiny fraction of those builds? Yes—by treating commit attribution
as an **optimal experimental design** problem guided by **Information Theory**.

---

## 2. Foundations of Information Theory: Measuring Uncertainty in Bits

Before we can ask an algorithm to "find the culprit with the least work," we
must mathematically define what **"information"** and **"uncertainty"**
actually mean.

Claude Shannon established this in 1948 by introducing four foundational
concepts: **Self-Information**, **Entropy**, **Conditional Entropy**, and
**Mutual Information**.

```
    UNIFORM UNCERTAINTY (H = 6.64 bits)                PINPOINTED CULPRIT (H = 0.05 bits)
  ┌─────────────────────────────────────┐            ┌─────────────────────────────────────┐
  │ 1/100  1/100  1/100 ... 1/100 1/100 │            │ 0.00   0.00   0.99 ... 0.00   0.00  │
  └─────────────────────────────────────┘            └─────────────────────────────────────┘
                    ▼                                                  ▲
                    └───────► Active Builds Destroy Entropy ───────────┘
```

### Concept 1: Self-Information (Surprise)
If an event has probability $p$, how much information does learning that the
event occurred provide?
$$I(x) = \log_2 \left(\frac{1}{p(x)}\right) = -\log_2 p(x) \quad \text{(measured in bits)}$$

* If you hear the sun rose today ($p \approx 1.0$), $I = -\log_2(1.0) = 0$ bits.
  You learned nothing new.
* If a coin lands on heads ($p = 0.5$), $I = -\log_2(0.5) = 1$ bit of
  information.
* If a 1-in-1,000 bug triggers ($p = 0.001$),
  $I = -\log_2(0.001) \approx 9.97$ bits.
  The rarer the event, the more "surprise" (information) it carries.

### Concept 2: Shannon Entropy (The Measure of Ignorance)
**Shannon Entropy $H(X)$** is the expected self-information across all possible
outcomes. It quantifies the total volume of uncertainty in a probability
distribution:
$$H(X) = -\sum_{x} P(x) \log_2 P(x)$$

Consider a suspect list of $N = 100$ commits, where $C$ is the true culprit
commit:
1. **Total Ignorance (Uniform Prior)**: When we know nothing, every commit is
   equally likely: $P(C = c) = 1/100$.
   $$H(C) = -\sum_{c=1}^{100} \frac{1}{100} \log_2 \left(\frac{1}{100}\right) = \log_2(100) \approx 6.64 \text{ bits}$$
2. **Absolute Certainty**: When we know with $100\%$ probability that commit 42
   is the culprit ($P(C=42)=1$, all others $0$):
   $$H(C) = -(1 \log_2 1 + 0) = 0 \text{ bits}$$
3. **Perplexity (Effective Suspects)**: The quantity $2^{H(C)}$ transforms
   entropy back into the intuitive number of "equally plausible suspects":
   $$\text{Perplexity} = 2^{H(C)}$$
   When $H(C) = 6.64$ bits, perplexity is $2^{6.64} = 100$ suspects. When
   $H(C) = 1.0$ bit, perplexity is $2^1 = 2$ suspects. When $H(C) \to 0$,
   perplexity is $1$ suspect.

### Concept 3: Joint Entropy (Total Uncertainty of a System)
What if our system is described by *two* related random variables, $X$ and $Y$
(for example, $X$ = candidate commit and $Y$ = test outcome)?

**Joint Entropy $H(X, Y)$** measures the total combined uncertainty of the pair
$(X, Y)$ taken together. It is defined as the expected self-information over the
joint probability distribution $P(x, y)$:
$$H(X, Y) = -\sum_{x} \sum_{y} P(x, y) \log_2 P(x, y)$$

**Core Properties of Joint Entropy**:
- **Independence Sum**: If $X$ and $Y$ are completely independent ($P(x, y) =
  P(x)P(y)$), their joint uncertainty is simply the sum of their individual
  entropies:
  $$H(X, Y) = H(X) + H(Y)$$
- **Subadditivity**: When $X$ and $Y$ share information, their joint uncertainty
  is strictly less than the sum of their separate uncertainties:
  $$H(X, Y) \le H(X) + H(Y)$$
  with equality if and only if $X$ and $Y$ are statistically independent.
- **The Chain Rule of Entropy**: Joint entropy decomposes into the uncertainty
  of the first variable plus the residual uncertainty of the second given the
  first:
  $$H(X, Y) = H(X) + H(Y \mid X) = H(Y) + H(X \mid Y)$$

### Concept 4: Conditional Entropy (Uncertainty Remaining After a Clue)
Suppose we have two related random variables, $X$ and $Y$. If someone reveals
the value of $X$, how much uncertainty remains about $Y$?

If we observe that $X$ took on a specific value $x$, the remaining entropy of
$Y$ is simply the Shannon entropy of the conditional distribution:
$$H(Y \mid X = x) = -\sum_{y} P(y \mid x) \log_2 P(y \mid x)$$

Because we do not know in advance which outcome $x$ will occur, **Conditional
Entropy $H(Y \mid X)$** is the *expected value* of this remaining uncertainty,
averaged across all possible outcomes of $X$:
$$H(Y \mid X) = \sum_{x} P(x) H(Y \mid X = x) = -\sum_{x} \sum_{y} P(x, y) \log_2 P(y \mid x)$$

From the Chain Rule, conditional entropy is also the slice of joint entropy
unaccounted for by $X$:
$$H(Y \mid X) = H(X, Y) - H(X)$$

**Physical Properties of Conditional Entropy**:
- **Independent variables**: If $X$ and $Y$ are completely independent, learning
  $X$ reveals nothing about $Y$:
  $$H(Y \mid X) = H(Y)$$
- **Deterministic dependence**: If $X$ completely determines $Y$ (e.g.
  $Y = f(X)$), no uncertainty remains once $X$ is known:
  $$H(Y \mid X) = 0 \text{ bits}$$
- **Information never increases uncertainty on average**: On average, observing
  evidence can only reduce (or preserve) your uncertainty, never increase it:
  $$0 \le H(Y \mid X) \le H(Y)$$

### Concept 5: Mutual Information (Information Gain & Shared Bits)
With individual, joint, and conditional entropies in hand, we can now formally
define what it means to **gain information**.

**Mutual Information $I(X; Y)$** measures the volume of information that one
random variable shares with another. It answers the fundamental question:
> *"How many bits of uncertainty about $Y$ are permanently eliminated by
> learning $X$?"*

From our entropy accounting, Mutual Information is the overlap between the two
variables—the amount by which their combined joint uncertainty is smaller than
the sum of their independent parts:
$$I(X; Y) = H(X) + H(Y) - H(X, Y)$$

Equivalently, it is the difference between our prior uncertainty about $Y$ and
our remaining uncertainty after observing $X$:
$$I(X; Y) = H(Y) - H(Y \mid X) = H(X) - H(X \mid Y)$$

We can also express Mutual Information directly across joint and marginal
probabilities:
$$I(X; Y) = \sum_{x} \sum_{y} P(x, y) \log_2 \left(\frac{P(x, y)}{P(x) P(y)}\right)$$

```text
  ┌─────────────────────────────────────────────────────────────┐
  │                 TOTAL JOINT ENTROPY H(X, Y)                 │
  │                                                             │
  │  ◄────────────── H(X) ──────────────►                       │
  │                     ◄─────────────── H(Y) ────────────────► │
  │  ┌──────────────────┬───────────────┬─────────────────────┐ │
  │  │                  │               │                     │ │
  │  │     H(X | Y)     │    I(X; Y)    │      H(Y | X)       │ │
  │  │                  │               │                     │ │
  │  │   Exclusive to   │    Shared     │    Exclusive to     │ │
  │  │    X without Y   │  information  │     Y without X     │ │
  │  │                  │   (overlap)   │                     │ │
  │  └──────────────────┴───────────────┴─────────────────────┘ │
  └─────────────────────────────────────────────────────────────┘
```

**Reading the Information Accounting Diagram**:
Think of the diagram above as partitioning the uncertainty space into three
distinct, non-overlapping segments:
1. **Left segment: $H(X \mid Y)$** is the private uncertainty unique to $X$ that
   $Y$ can never explain (e.g. random hardware jitter during a test run).
2. **Middle segment: $I(X; Y)$** is the shared Mutual Information—the overlap
   conveyed by both variables.
3. **Right segment: $H(Y \mid X)$** is the private uncertainty unique to $Y$
   that $X$ can never explain.

Notice how the total quantities are constructed:
* **$H(X)$ (Total uncertainty of $X$)** spans the **left + middle** segments:
  $$H(X) = H(X \mid Y) + I(X; Y)$$
* **$H(Y)$ (Total uncertainty of $Y$)** spans the **middle + right** segments:
  $$H(Y) = I(X; Y) + H(Y \mid X)$$
* **$H(X, Y)$ (Joint Entropy)** is the entire outer box enclosing all three
  segments:
  $$H(X, Y) = H(X \mid Y) + I(X; Y) + H(Y \mid X)$$

This geometric relationship makes the subtraction intuitive:
$$H(Y \mid X) = H(Y) - I(X; Y) \iff I(X; Y) = H(Y) - H(Y \mid X)$$
When you run a prospective test trial $X$, you capture the shared intersection
$I(X; Y)$, successfully shrinking your remaining uncertainty about the culprit
$Y$ down to just the unshared segment $H(Y \mid X)$!

**Core Properties of Mutual Information**:
1. **Non-Negativity**: $I(X; Y) \ge 0$, with equality $I(X; Y) = 0$ if and only
   if $X$ and $Y$ are statistically independent.
2. **Symmetry**: $I(X; Y) = I(Y; X)$.
3. **Self-Information**: $I(X; X) = H(X)$ (the information a variable shares
   with itself is its own entropy).
4. **Upper Bound**: $I(X; Y) \le \min(H(X), H(Y))$ (you cannot learn more bits
   about $Y$ than $Y$ possesses).

> [!IMPORTANT]
> **The Active Learning Mandate**: In flaky test attribution, Mutual Information
> $I(\text{Trial at } r; C)$ represents the **expected reduction in Shannon
> entropy about the culprit commit $C$** obtained by executing a test trial at
> candidate revision $r \in \{0, \dots, N-1\}$.
>
> At every step of our search, we select the revision $r^*$ that
> **maximizes $I(\text{Trial}; C)$**:
> $$r^* = \arg\max_{r} \Big( H(\text{Trial at } r) - H(\text{Trial at } r \mid C) \Big)$$
> This guarantees that every expensive CI build destroys the maximum possible
> volume of uncertainty!

---

## 3. The Bayesian Changepoint Mechanism: The $\mathcal{O}(N)$ Prefix Sweep

To compute entropy and information gain, we first need an exact, fast way to
calculate the posterior probability distribution $P(C = c \mid \mathcal{D})$
across all candidate revisions $c \in \{0, \dots, N-1\}$ given our observed test
data $\mathcal{D}$.

### The Changepoint Model
In version control history, commits are ordered sequentially:
$$0, 1, 2, \dots, N-1$$
Our test history across these $N$ commits forms our observed dataset:
$$\mathcal{D} = \{(F_i, P_i)\}_{i=0}^{N-1}$$
where:
* $F_i \in \{0, 1, 2, \dots\}$ is the count of observed test **Failures** at
  commit index $i$.
* $P_i \in \{0, 1, 2, \dots\}$ is the count of observed test **Passes** at
  commit index $i$.

We hypothesize that a single commit $c$ introduced the flake regression:
* **Healthy Regime (Before Culprit)**: For all commits strictly before $c$
  ($i < c$), test failures are generated by an unknown baseline flake rate
  $p_{\text{before}}$ (e.g. $1\%$ infra noise).
* **Broken Regime (After Culprit)**: For all commits at or after $c$
  ($i \ge c$), test failures are generated by an elevated failure rate
  $p_{\text{after}}$ (e.g. $25\%$ race condition).

Candidate commit $c$ cuts the repository into two disjoint halves:
1. The **Pre-Culprit Prefix**: Commits $0 \le i < c$, with total failures
   $F_{<c} = \sum_{i < c} F_i$ and total passes $P_{<c} = \sum_{i < c} P_i$.
2. The **Post-Culprit Suffix**: Commits $c \le i < N$, with total failures:
   $$F_{\ge c} = \sum_{i \ge c} F_i, \quad P_{\ge c} = \sum_{i \ge c} P_i$$

### Why "Integrate Out" Unknown Rates? (The Nuisance Parameter Problem)
Our ultimate goal is to find **$c$**—the index of the commit that introduced
the regression. Under Bayes' Rule, the posterior distribution over candidate
culprits is:
$$P(C = c \mid \mathcal{D}) \propto P(\mathcal{D} \mid C = c) \cdot P(C = c)$$

To evaluate the likelihood $P(\mathcal{D} \mid C = c)$, a statistical model
needs to know the probability of test failure in the healthy regime
($p_{\text{before}}$) and in the broken regime ($p_{\text{after}}$).

**The Challenge**: In real-world debugging, **we do not know either rate**.
Was the background flake rate $0.5\%$ or $2\%$? Did the culprit bump the flake
rate to $15\%$, $25\%$, or $50\%$? We only observe raw counts of passes and
failures.

In statistics, $p_{\text{before}}$ and $p_{\text{after}}$ are **nuisance
parameters**: variables that the likelihood function requires, but whose exact
numerical values are not the object of our investigation.

#### Why Not Estimate the Rates with Maximum Likelihood (MLE)?
You might ask: *"Why not simply plug in point estimates
$\hat{p} = \frac{F}{F+P}$?"*
1. **Small-sample collapse**: Early in a bisect, candidate revisions may only
   have 1 or 2 test runs. If a revision ran once and failed, $\hat{p} = 1.0$.
   If that commit later passes, a model assuming $\hat{p} = 1.0$ assigns
   likelihood zero, breaking the calculation.
2. **Hypothesis coupling**: For every candidate culprit $c$, fitting separate
   MLE rates $\hat{p}_{<c}$ and $\hat{p}_{\ge c}$ causes candidate splits with
   sparse data to wildly overfit, creating artificial spikes in likelihood.

#### The Bayesian Solution: Marginalization
Instead of fixing a single point estimate, Bayesian inference dictates that we
**integrate out (marginalize) the unknown rates**. We evaluate the likelihood
under *every possible failure rate* $p \in [0, 1]$, weighted by our prior
belief:
$$P(F, P) = \int_0^1 \underbrace{p^F (1-p)^P}_{\text{Likelihood}}
  \cdot \underbrace{P(p)}_{\text{Prior}} \, dp$$

When we calculate this integral, **the unknown parameter $p$ completely
disappears from the equation!** What remains is the **Marginal Likelihood**
(Bayesian Evidence): a single, exact number quantifying how well candidate
split $c$ explains our observed test counts across all possible flake rates.

#### The Magic of Beta-Bernoulli Conjugacy
Usually, integrating out parameters over continuous dimensions requires slow
numerical integration or Markov Chain Monte Carlo (MCMC).

However, because the Bernoulli likelihood and the Beta prior are
**conjugate** (as established in **Sheet 1**), this integral has an exact
closed-form solution: **Euler's Beta function** $B(\alpha, \beta)$.

Assuming an uninformative uniform prior $p \sim \text{Beta}(1, 1)$ (where
$P(p) = 1$):
$$P(F, P) = \int_0^1 p^F (1-p)^P \, dp
  = B(F + 1, P + 1)
  = \frac{F! \cdot P!}{(F + P + 1)!}$$

*(Notation note: The outer $P(\dots)$ denotes probability, while the inner
argument $P$ denotes the count of test Passes.)*

In log-space (to avoid numerical underflow):
$$\ln P(F, P) = \ln \Gamma(F+1) + \ln \Gamma(P+1) - \ln \Gamma(F+P+2)
  = \text{betaln}(F+1, P+1)$$

### The Double-Sweep Algorithm: $\mathcal{O}(N)$ Exact Inference
Because the pre-culprit and post-culprit regimes are conditionally independent
given changepoint $c$, the total marginal likelihood of the entire dataset
$\mathcal{D}$ under hypothesis $C = c$ is simply the product of their separate
marginal likelihoods:
$$P(\mathcal{D} \mid C = c) = P(F_{<c}, P_{<c}) \times P(F_{\ge c}, P_{\ge c})$$

Taking logarithms converts this product into a sum of two log-Beta evaluations:
$$\ln P(\mathcal{D} \mid C = c) = \text{betaln}(F_{<c} + 1, P_{<c} + 1) + \text{betaln}(F_{\ge c} + 1, P_{\ge c} + 1)$$

```text
  Index:     0     1   ...   c-1        │   c     c+1   ...    N-1
  Commits: [c_0] [c_1] ... [c_{c-1}]    │ [c_c] [c_{c+1}] ... [c_{N-1}]
  ──────────────────────────────────────┼──────────────────────────────────────
              BEFORE CULPRIT            │            AFTER CULPRIT
        F_{<c} fails, P_{<c} passes     │     F_{>=c} fails, P_{>=c} passes
   Marginal: betaln(F_{<c}+1, P_{<c}+1) │ Marginal: betaln(F_{>=c}+1, P_{>=c}+1)
```

**The $\mathcal{O}(N^2) \to \mathcal{O}(N)$ Computational Breakthrough**:
- **Naive evaluation**: Summing $(F_{<c}, P_{<c})$ and $(F_{\ge c}, P_{\ge c})$
  from scratch for each candidate $c$ takes $\mathcal{O}(N)$ per candidate,
  requiring $\mathcal{O}(N^2)$ time total across the commit history.
- **Double-Sweep optimization**: We precompute prefix sums in a single pass from
  left to right ($0 \to N-1$) and suffix sums in a single pass from right to
  left ($N-1 \to 0$):
  1. **Forward Cumulative Sweep (`fwd[c]`)**:
     $$\text{fwd}[c] \equiv \text{betaln}(F_{<c} + 1, P_{<c} + 1)$$
  2. **Backward Cumulative Sweep (`bwd[c]`)**:
     $$\text{bwd}[c] \equiv \text{betaln}(F_{\ge c} + 1, P_{\ge c} + 1)$$

Every candidate changepoint $c$ can now be evaluated in $\mathcal{O}(1)$ time!
We then normalize across all $N$ candidate commits using `logsumexp`:
$$P(C = c \mid \mathcal{D}) = \frac{\exp(\text{fwd}[c] + \text{bwd}[c])}{\sum_{j=0}^{N-1} \exp(\text{fwd}[j] + \text{bwd}[j])}$$

No Markov Chain Monte Carlo (MCMC), no sampling error, and no numerical
approximations—this produces the **exact closed-form posterior distribution** in
milliseconds.

---

## 4. Active Learning: The Maximum Information Gain Selector

Now that we have computed the posterior distribution $P(C = c \mid \mathcal{D})$
from our existing test history $\mathcal{D}$, we face the central active
engineering challenge:
> **Which commit should we build and test next?**

In sequential Bayesian analysis, *"today's posterior is tomorrow's prior."* Our
current posterior $P(C = c \mid \mathcal{D})$ represents our complete state of
knowledge before we run any new prospective trials. For convenience, we denote
our current belief state as:
$$P(C = c) \equiv P(C = c \mid \mathcal{D})$$

Instead of testing commits blindly or relying on naive binary search, we treat
test execution as an **optimal experiment design** problem guided by
Information Theory: we run a batch of $M$ builds at whichever revision $r^*$
yields the **maximum expected Mutual Information** $I(\text{Trial at } r; C)$.

### Step 1: Posterior-Weighted Expected Failure Rates
To predict what a test trial at revision $r$ will produce, we first estimate the
underlying failure rates before and after the changepoint.

Under candidate changepoint $c$, the expected failure rate before $c$ is
given by **Laplace's Rule of Succession** (the posterior mean of Beta-Binomial
conjugacy from Sheet 1):
$$\mathbb{E}[p \mid F, P] = \frac{F + 1}{F + P + 2}$$

Because we have uncertainty about which commit $c$ is the true culprit, we take
the posterior-weighted expectation across all $N$ candidate changepoints:
$$\hat{p}_{\text{before}} = \sum_{c=0}^{N-1} P(C = c \mid \mathcal{D}) \cdot \frac{F_{<c} + 1}{F_{<c} + P_{<c} + 2}$$
$$\hat{p}_{\text{after}} = \sum_{c=0}^{N-1} P(C = c \mid \mathcal{D}) \cdot \frac{F_{\ge c} + 1}{F_{\ge c} + P_{\ge c} + 2}$$

These two scalar rates represent our current Bayesian consensus on the baseline
flake rate ($\hat{p}_{\text{before}} \approx 1\%$) and the regressed flake rate
($\hat{p}_{\text{after}} \approx 25\%$).

### Step 2: The Mixture Distribution of a Prospective Test
Suppose we allocate compute quota to run a batch of $M$ test builds at candidate
revision $r$. What outcomes should we expect?

Before running the builds, we do not know whether revision $r$ is healthy or
broken:
* **Case 1: Culprit is at or before $r$ ($C \le r$)**:
  Revision $r$ is *broken*. The number of test failures will follow:
  $$\text{Binomial}(M, \hat{p}_{\text{after}})$$
  Our current belief that Case 1 holds is the cumulative posterior weight:
  $$w_{\text{after}} \equiv P(C \le r \mid \mathcal{D}) = \sum_{c=0}^{r} P(C = c \mid \mathcal{D})$$

* **Case 2: Culprit is strictly after $r$ ($C > r$)**:
  Revision $r$ is *healthy*. The number of test failures will follow:
  $$\text{Binomial}(M, \hat{p}_{\text{before}})$$
  Our current belief that Case 2 holds is the complement probability:
  $$w_{\text{before}} \equiv P(C > r \mid \mathcal{D}) = 1 - w_{\text{after}}$$

The predicted outcome distribution of our prospective trial over possible
failure counts $k \in \{0, 1, \dots, M\}$ is the weighted mixture:
$$P(\text{Trial} = k \mid \mathcal{D}) = w_{\text{before}} \cdot \text{Binomial}(k; M, \hat{p}_{\text{before}}) + w_{\text{after}} \cdot \text{Binomial}(k; M, \hat{p}_{\text{after}})$$

### Step 3: Isolating Signal from Noise via Mutual Information
How much will observing this prospective trial actually teach us about the
culprit commit $C$?

1. **Total Predicted Uncertainty $H(\text{Trial} \mid \mathcal{D})$**:
   The Shannon entropy of the mixture distribution measures how uncertain we are
   about the outcome of the $M$ builds:
   $$H(\text{Trial} \mid \mathcal{D}) = -\sum_{k=0}^{M} P(\text{Trial} = k \mid \mathcal{D}) \log_2 P(\text{Trial} = k \mid \mathcal{D})$$
   When $w_{\text{before}} \approx w_{\text{after}} \approx 0.5$ (maximum
   ignorance about whether revision $r$ is broken), the mixture is bimodal and
   $H(\text{Trial} \mid \mathcal{D})$ is maximized.

2. **Inherent Flake Noise $H(\text{Trial} \mid C, \mathcal{D})$**:
   Even if an oracle told us whether revision $r$ was healthy or broken, the
   test is still intrinsically flaky! Random infrastructure jitter produces
   Bernoulli coin-flip variance that can never be attributed to the culprit:
   $$H(\text{Trial} \mid C, \mathcal{D}) = w_{\text{before}} H(\text{Binomial}(M, \hat{p}_{\text{before}})) + w_{\text{after}} H(\text{Binomial}(M, \hat{p}_{\text{after}}))$$

3. **Mutual Information (Pure Information Gain)**:
   By subtracting the unavoidable flake noise from the total trial uncertainty,
   **Mutual Information isolates the pure signal about the culprit $C$**:
   $$I(\text{Trial at } r; C \mid \mathcal{D}) = H(\text{Trial} \mid \mathcal{D}) - H(\text{Trial} \mid C, \mathcal{D})$$

```text
  Information Gain I(r; C) across Revisions
   Bits
   ▲
   │                 Peak Information Gain: Choose r*!
   │                       ┌───▲───┐
   │                      ┌┘       └┐
   │                     ┌┘         └┐
   │                    ┌┘           └┐
   │         ───────────┘             └───────────
   └────────────────────────────────────────────────► Revision r
             Commits 0..30    Commits 40..55     Commits 70..100
             (Already Known   (Maximum Ignorance  (Already Known
              to be Before)     Boundary)          to be After)
```

At each active iteration, the search engine evaluates $I(\text{Trial at } r; C)$
for all candidate revisions $r \in \{0, \dots, N-1\}$ and dispatches tests to
the revision that maximizes information gain:
$$r^* = \arg\max_{r} I(\text{Trial at } r; C \mid \mathcal{D})$$

### Why Information Gain Outperforms Human Intuition
The mutual information selector automatically balances three critical
engineering dynamics:
1. **Dynamic Binary Search (Exploration)**: When posterior uncertainty is spread
   widely across the commit range, $I(r; C)$ naturally peaks near the median of
   $P(C)$, behaving like an optimal probabilistic binary search.
2. **Boundary Focusing (Exploitation)**: When uncertainty narrows down to a few
   suspect commits (e.g. commit 41 vs 42), the information landscape
   concentrates tightly at the boundary, repeatedly testing that exact commit
   until the evidence crosses our confidence threshold ($P \ge 0.99$).
3. **Immunity to Over-Testing (Noise Satiation)**: If a commit has already been
   tested 50 times, running 5 more tests yields virtually zero new information
   ($I \to 0$). The algorithm naturally moves on to undertested regions without
   needing hardcoded heuristic rules!

---

## 5. Master Architecture & Decision Matrix

| Strategy | Deterministic Regressions ($0 \to 1$) | Flaky Regressions ($1\% \to 25\%$) | Total Builds Required | Failure Mode |
| :--- | :--- | :--- | :--- | :--- |
| **Git Bisect** | **Optimal** ($\mathcal{O}(\log N)$) | **Catastrophic** | $7 - 10$ builds | Single accidental pass discards true culprit with $>70\%$ probability. |
| **Grid / Uniform Sweep** | Excessive | Reliable but wasteful | $M \times N$ builds (e.g., $100 \times 100 = 10{,}000$) | Prohibitive compute quota and engineering turnaround time. |
| **Active Bayesian Search** | **Optimal** ($\mathcal{O}(\log N)$) | **Robust & Optimal** ($\mathcal{O}(\frac{\log N}{D_{\text{KL}}})$) | **Minimal** ($30 - 80$ builds total) | None: Guaranteed convergence to posterior confidence threshold ($P \ge 0.99$). |

*Note on sample complexity*: In the table above,
$D_{\text{KL}} \equiv D_{\text{KL}}(p_{\text{after}} \parallel p_{\text{before}})$
denotes the Kullback-Leibler divergence between the post-culprit and baseline
failure distributions. It measures how statistically distinguishable the flake
rate is from background noise. When the regression delta is small,
$D_{\text{KL}}$ is small, naturally requiring more builds to achieve certainty.

---

## 6. Real-World Architecture: Production Active Culprit Search Engine

In modern continuous integration environments, this exact mathematics powers
autonomous flake attribution:
1. **`init`**: Ingests historical pass/fail test execution records from CI build
   telemetry and maps git revisions into contiguous indices.
2. **`get_culprit_dist`**: Runs the forward/backward Beta-Bernoulli sweep to
   produce $P(C = c)$, Shannon entropy $H(C)$, and perplexity.
3. **`find_best_revision_to_test`**: Evaluates mutual information
   $I(\text{Trial}; C)$ across all revisions for an $M$-run batch.
4. **`launch_test_run`**: Automatically triggers remote CI test runs pinned to
   revision $r^*$.
5. **Convergence**: Once any commit hits $P(C = c) \ge 0.99$, the search
   terminates, reporting the culprit commit, 99% credible interval, and
   empirical flake delta:
   $$\Delta p = \hat{p}_{\text{after}} - \hat{p}_{\text{before}}$$

---

### 🐍 Next Step: See the Code in Action!
Navigate to the executable companion lab: **[Python Track: Appendix F](../python/appendix_information_theory_and_culprit_search.ipynb)** or **[R Track: Appendix F](../r/appendix_information_theory_and_culprit_search.ipynb)** to build and simulate the entire active learning loop from scratch!
